# 01 · Exploratory Analysis

First-look statistics on the Migration Atlas seed graph. This notebook is the "is the data sane?" sanity check before any modeling. Run from the repo root:

```bash
make setup
jupyter lab notebooks/01_eda.ipynb
```

What we cover:
1. Load the knowledge graph
2. Distribution of node kinds and edge kinds
3. Top countries by foreign-born population
4. Connectivity: degree distribution, clustering
5. Era breakdown of country origins
6. Sanity checks — no dangling edges, no duplicate ids

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter

from migration_atlas.graph.build import build_default

backend = build_default()
G = backend.graph
print(f'Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}')

## 1. Distribution of node kinds

In [ ]:
kinds = Counter(G.nodes[n]['kind'] for n in G.nodes)
pd.Series(kinds).sort_values().plot(kind='barh', color='#c2410c', figsize=(7, 3))
plt.title('Nodes by kind')
plt.xlabel('Count')
plt.tight_layout()
plt.show()
kinds

## 2. Top countries by foreign-born population

In [ ]:
country_data = []
for n in G.nodes:
    attrs = G.nodes[n]
    if attrs.get('kind') == 'country':
        country_data.append({
            'country': attrs.get('name'),
            'foreign_born_us': attrs.get('foreign_born_us', 0),
            'era': attrs.get('era'),
        })
df = pd.DataFrame(country_data).sort_values('foreign_born_us', ascending=False)
df.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
df.head(15).set_index('country')['foreign_born_us'].plot(
    kind='barh', color='#c2410c', ax=ax
)
ax.set_title('Top 15 countries by foreign-born population in U.S.')
ax.set_xlabel('Foreign-born population')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Edge type distribution

In [ ]:
edge_kinds = Counter(d['kind'] for u, v, d in G.edges(data=True))
pd.Series(edge_kinds).sort_values().plot(kind='barh', color='#1e3a8a', figsize=(7, 3))
plt.title('Edges by kind')
plt.xlabel('Count')
plt.tight_layout()
plt.show()
edge_kinds

## 4. Degree distribution

In [ ]:
degrees = dict(G.degree())
deg_df = pd.DataFrame([
    {'node': n, 'degree': d, 'kind': G.nodes[n]['kind'], 'name': G.nodes[n].get('name')}
    for n, d in degrees.items()
]).sort_values('degree', ascending=False)
deg_df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
deg_df.head(15).set_index('name')['degree'].plot(kind='barh', color='#581c87', ax=ax)
ax.set_title('Top 15 most-connected nodes')
ax.set_xlabel('Degree')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

Observation: the most-connected nodes are typically (a) policy laws like the 1965 INA that touched many countries, and (b) visa categories used by many origin groups. This matches what we'd expect from a domain-correct graph.

## 5. Era breakdown

In [ ]:
era_pop = df.groupby('era')['foreign_born_us'].sum().sort_values(ascending=False)
era_pop.plot(kind='bar', color=['#c2410c', '#1e3a8a', '#115e59'], figsize=(7, 4))
plt.title('Foreign-born population by era of first major wave')
plt.xlabel('Era')
plt.ylabel('Population')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
era_pop

## 6. Sanity checks

In [ ]:
# No dangling edges
node_ids = set(G.nodes)
dangling = [(u, v) for u, v in G.edges() if u not in node_ids or v not in node_ids]
print(f'Dangling edges: {len(dangling)}')
assert not dangling, 'Found dangling edges!'

# No duplicate ids (NetworkX prevents this, but check just in case)
assert len(node_ids) == G.number_of_nodes()

# Strongly-connected components
scc_count = nx.number_strongly_connected_components(G)
wcc_count = nx.number_weakly_connected_components(G)
print(f'Strongly connected components: {scc_count}')
print(f'Weakly connected components:   {wcc_count}')

## Takeaways for the case study

- The seed graph has the expected shape: a few high-degree hub nodes (1965 INA, family-based visa) and a long tail.
- The modern era dominates by absolute population — almost entirely a consequence of the 1965 INA reopening immigration from Asia and Latin America.
- The graph is weakly connected (all reachable if we ignore edge direction) but has many SCCs, which makes sense — laws "point at" countries and visas without reciprocal edges.

Next: `02_stance_training.ipynb` walks through training the stance classifier.